Pretraining

In [1]:
!pip install -q transformers datasets evaluate accelerate sentencepiece scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [5]:
english_df = pd.read_csv("/content/English_train_data.csv")
hindi_df = pd.read_csv("/content/final_dataset_hindi1.csv")
bengali_df = pd.read_csv("/content/merged_bengali_dataset.csv")
hindi_trans_df = pd.read_csv("/content/english_to_hindi.csv")
bengali_trans_df = pd.read_csv("/content/english_to_bengali.csv")

print(english_df.head())
print(hindi_df.head())
print(bengali_df.head())
print(hindi_trans_df.head())
print(bengali_trans_df.head())

                                            sentence    label
0  : If we are going to slow climate change, we m...    FAVOR
1  World leaders duped by manipulated global warm...  AGAINST
2  To be sure the American greens have greater am...     NONE
3  Dealing with global warming is an internationa...    FAVOR
4  : Concussion is like climate change It isn't r...  AGAINST
                                                text
0  सूर्य ग्रहण: दिन में चार मिनट तक रहेगा अंधेरा,...
1  भारतीय महिला बैडमिंटन टीम ने यंग ब्रिगेड के सह...
2  नेपाल में बोधिचित्त वृक्षों की चिंता: 'सोने की...
3  आईपीएल के सुपर संडे में कई रिकॉर्ड बने, लेकिन ...
4  भारत ईरान ने मतभेद भुलाकर किया ये अहम समझौता, ...
                                                text
0  মার্কিন মহাকাশ সংস্থা নাসা আয়োজিত বিশ্বের বৃহ...
1  বঙ্গবন্ধুর জন্ম শতবর্ষ উপলক্ষে সারাদেশে ১ কোটি...
2  জার্মানির আন্তর্জাতিক সম্প্রচার কেন্দ্র ডয়চে ভ...
3  অর্থমন্ত্রী আ হ ম মুস্তফা কামাল বলেছেন, পেঁয়াজ...
4  কাপ্তাই হ্রদের দূষণ দূর করতে অভিযান পরিচাল

In [6]:
english = english_df["sentence"].astype(str)

hindi = hindi_df["text"].astype(str)

bengali = bengali_df["text"].astype(str)

hindi_trans = hindi_trans_df["hindi"].astype(str)

bengali_trans = bengali_trans_df["bengali"].astype(str)

In [7]:
all_text = pd.concat(
    [
        english,
        hindi,
        bengali,
        hindi_trans,
        bengali_trans
    ],
    ignore_index=True
)

all_text = all_text.dropna()

all_text = all_text.drop_duplicates()

all_text = all_text.str.strip()

all_text = all_text[all_text!=""]

print("Total sentences:",len(all_text))

Total sentences: 46984


In [8]:
mlm_df = pd.DataFrame()

mlm_df["text"] = all_text

mlm_df.head()

,text
0,": If we are going to slow climate change, we m..."
1,World leaders duped by manipulated global warm...
2,To be sure the American greens have greater am...
3,Dealing with global warming is an internationa...
4,: Concussion is like climate change It isn't r...


In [9]:
train_df, valid_df = train_test_split(
    mlm_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print(train_df.shape)
print(valid_df.shape)

(42285, 1)
(4699, 1)


In [10]:
train_dataset = Dataset.from_pandas(train_df)

valid_dataset = Dataset.from_pandas(valid_df)

In [11]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

In [12]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [13]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_valid = valid_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/42285 [00:00<?, ? examples/s]

Map:   0%|          | 0/4699 [00:00<?, ? examples/s]

In [14]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [15]:
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
from transformers import EarlyStoppingCallback

In [17]:
training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/XLMR_MLM8",

    num_train_epochs=5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_steps=159, # Calculated from 0.06 * (len(train_df) / per_device_train_batch_size / gradient_accumulation_steps * num_train_epochs)

    fp16=True,

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=1,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    report_to="none"

)

In [18]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    data_collator=data_collator,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]

)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.601623,1.562832
2,3.301255,1.520925
3,3.095786,1.466535
4,3.087666,1.440445
5,3.095471,1.422359


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


TrainOutput(global_step=13215, training_loss=3.2894143729872023, metrics={'train_runtime': 9286.228, 'train_samples_per_second': 22.768, 'train_steps_per_second': 1.423, 'total_flos': 4.51237915269504e+16, 'train_loss': 3.2894143729872023, 'epoch': 5.0})

In [20]:
SAVE_PATH="/content/drive/MyDrive/XLM-R Cross Validation/Best_XLMR_MLM8"

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("Saved Successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Successfully!


In [21]:
import os

os.listdir(SAVE_PATH)

['config.json',
 'model.safetensors',
 'tokenizer_config.json',
 'tokenizer.json',
 'training_args.bin']

In [22]:
from transformers import AutoTokenizer,AutoModelForMaskedLM

tokenizer=AutoTokenizer.from_pretrained(SAVE_PATH)

model=AutoModelForMaskedLM.from_pretrained(SAVE_PATH)

print("Loaded Successfully")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loaded Successfully
